### AG News preprocessing
### Load the raw parquet files, inspect schema, merge records, and clean text while preserving meaning.


In [ ]:
import json
from pathlib import Path

import pandas as pd

raw_dir = Path('..') / 'data' / 'raw'
processed_dir = Path('..') / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

# Read every parquet file and combine them into a single DataFrame.
parquet_files = sorted(raw_dir.glob('*.parquet'))
parquet_files


In [ ]:
df_list = [pd.read_parquet(f) for f in parquet_files]
df = pd.concat(df_list, ignore_index=True)

print(f'Parquet files: {len(parquet_files)}')
print(f'Combined rows: {len(df)}')
print('Columns:', list(df.columns))
print(df.head(3).to_dict(orient='records'))


In [ ]:
# Clean only whitespace and escaped characters without changing wording.
def normalize_text(value):
    if pd.isna(value):
        return None
    text = str(value)
    # Replace escaped line breaks/tabs with spaces, then collapse repeated whitespace.
    text = text.replace('\\n', ' ').replace('\\r', ' ').replace('\\t', ' ')
    text = ' '.join(text.split())
    return text if text else None

# Keep category label as metadata.
df = df.rename(columns={'label': 'category'})

original_rows = len(df)
# Remove empty/null text and exact duplicates on the cleaned text, keeping first occurrence.
cleaned = df[['text', 'category']].copy()
cleaned['text'] = cleaned['text'].map(normalize_text)
cleaned = cleaned.dropna(subset=['text']).drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)

removed_empty = original_rows - len(cleaned)
print(f'Original rows: {original_rows}')
print(f'Rows after empty/drop duplicates: {len(cleaned)}')
print(f'Rows removed: {removed_empty}')
print(cleaned.head(3).to_dict(orient='records'))


In [ ]:
# Generate a unique integer ID for each document.
cleaned['id'] = list(range(1, len(cleaned) + 1))
cleaned = cleaned[['id', 'text', 'category']]

# Save the final cleaned dataset as JSON Lines.
records = [
    {
        'id': int(row['id']),
        'text': row['text'],
        'metadata': {'category': row['category']}
    }
    for _, row in cleaned.iterrows()
]

output_path = processed_dir / 'documents.jsonl'
with output_path.open('w', encoding='utf-8') as f:
    for item in records:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'Wrote {len(records)} records to {output_path}')


In [ ]:
# Create a reproducible 1,000-record dev sample.
# Keep deterministic ordering by using the cleaned dataset order.
random_seed = 42
sample_size = 1000
if len(cleaned) > sample_size:
    sample_df = cleaned.sample(n=sample_size, random_state=random_seed).reset_index(drop=True)
else:
    sample_df = cleaned.reset_index(drop=True)

sample_records = [
    {
        'id': int(row['id']),
        'text': row['text'],
        'metadata': {'category': row['category']}
    }
    for _, row in sample_df.iterrows()
]

sample_path = processed_dir / 'documents_dev.jsonl'
with sample_path.open('w', encoding='utf-8') as f:
    for item in sample_records:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'Wrote {len(sample_records)} dev records to {sample_path}')


In [ ]:
# Print summary metrics and a few examples for validation.
final_df = cleaned.copy()
final_df['category'] = final_df['category'].astype(str)

category_distribution = final_df['category'].value_counts().sort_index()
print('Final record count:', len(final_df))
print('Duplicate count:', original_rows - len(final_df))
print('Category distribution:')
print(category_distribution)
print('\nExamples:')
for _, row in final_df.head(3).iterrows():
    print({'id': int(row['id']), 'category': row['category'], 'text': row['text'][:180]})
